In [1]:
import torch
from torch.utils.data import TensorDataset, ConcatDataset
from torch import Tensor
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import os
from load_store_utils import load_stats, resume_model
from generate import X, U, DU, D2U, PDE_KEYS, PDE_VALUES, IC_KEYS, IC_VALUES, TIMES
from typing import List, Tuple
from model import PdeNet
from data_utils import extract_TensorDataset, include_time_in_input, extract_targets, extract_boundary, extract_interior
import matplotlib.ticker as ticker
from pde_utils import key_str, ic_key_str

In [2]:
PHY_SYS = "Diffusion" # Diffusion ReactionDiffusion AdvectionDiffusion
PLOT = False
# ------------------------------------------------------------
DATA_DIR = "AdvectionReactionDiffusion"
if PHY_SYS == "Diffusion":
    DATA_DIR += "/Heat"
elif PHY_SYS == "ReactionDiffusion":
    DATA_DIR += "/AllenCahn"
elif PHY_SYS == "AdvectionDiffusion":
    DATA_DIR += "/Convection"
else:
    raise ValueError(f"Unknown physical system '{PHY_SYS}'.")

DATA_DIR += "/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_0-9/rep0"
FULL_DATASET = DATA_DIR + "/full_datasets.pth"
TEST_INDICES = DATA_DIR + "/intra_test_indices.pth"

In [3]:
def latex_sci(value, ub = None):
    v = value
    value = "{:.2e}".format(value)
    base, exponent = value.split("e")
    # remove leading zeros and plus signs from exponent
    exponent = int(exponent)
    if ub is not None and v < ub:
        return r"$\mathbf{" + str(base) + r" \times 10^{" + str(exponent) + r"}}$"
    return r"$" + str(base) + r" \times 10^{" + str(exponent) + r"}$"

def plot_points(
        pointss: List[Tensor],
        labelss: List[Tensor],
        nrows: int,
        ncols: int,
        label_name: str = "u",
        include_differences: bool = False,
        include_abs_differences: bool = False,
        save: bool = False,
        dst_file: str = None,
        cmap: str = "inferno",
        title: str = "",
        subtitles: list = [],
        figsize: tuple = (5, 5),
        vmin: float = None,
        vmax: float = None
    ) -> None:

    #if type(dataset) is str:
    #    datasets = [torch.load(dataset, weights_only=False)]
    #elif type(dataset) is list:
    #    datasets = []
    #    for ds in dataset:
    #        if type(ds) is str:
    #            datasets.append(torch.load(ds, weights_only=False))
    #        else:
    #            datasets.append(ds)
    #else:
    #    datasets = [dataset]

    font = {'size': 10}
    matplotlib.rc('font', **font)
    fig = plt.figure(figsize=figsize)

    gs = GridSpec(nrows, ncols, figure=fig)
    axes = []
    for j in range(ncols):
        for i in range(nrows):
            axes.append(fig.add_subplot(gs[i, j]))
    
    if subtitles == []:
        subtitles = ["" for _ in axes]
    for ax, points, labels, subtitle in zip(axes, pointss, labelss, subtitles):
        if vmin is not None and vmax is not None:
            scatter = ax.scatter(points[:, 0], points[:, 1], s=1, alpha=1, c=labels.flatten(), cmap=cmap, vmin=vmin, vmax=vmax)
        else:
            scatter = ax.scatter(points[:, 0], points[:, 1], s=1, alpha=1, c=labels.flatten(), cmap=cmap)

        #if ax == axes[-1]:
        #    cbar = fig.colorbar(scatter, ax=ax)
        #    cbar.set_label(name)
        #ax.set_xlabel('x')
        #ax.set_ylabel('y')
        if subtitle != "":
            ax.set_title(subtitle)
        
        ax.set_xticks([-1, 0, 1])
        ax.set_yticks([-1, 0, 1])
        ax.plot()
    fig.subplots_adjust(right=0.85, wspace=0.5, hspace=0.5) 
    cbar_ax = fig.add_axes([0.88, 0.15, 0.01, 0.7]) # [left, bottom, width, height]
    fig.colorbar(scatter, cax=cbar_ax, label=label_name)
    #cbar = fig.colorbar(scatter, ax=axes, fraction=0.046, pad=0.04)
    #cbar.set_label(names[-1])
    
    #plt.gca().set_aspect('equal', adjustable='box')
    fig.suptitle(title)
    #fig.tight_layout()
    #plt.tight_layout()
    if save:
        plt.savefig(fname=dst_file, bbox_inches='tight', dpi=300)
    plt.show()

In [4]:
if PLOT:
    samples = [0, 1, 2, 3, 4]
    ts = [1, 3, 6, 9]
    u = DU # U DU D2U
    # -------------
    
    if u == U:
        label_name = r"$u$"
    elif u == DU:
        label_name = r"$\frac{du}{dt}$"
    elif u == D2U:
        label_name = r"$\frac{d^2u}{dt^2}$"
    else:
        raise ValueError(f"Invalid label index {u}.")
    
    for sample in samples:
        titles = f"sample {sample}\n"
        datasets = torch.load(FULL_DATASET, weights_only=False).datasets[sample]
        datasets = [datasets.datasets[t] for t in ts]
        pointss = [ds.tensors[X][:] for ds in datasets]
        if u == U:
            labelss = [ds.tensors[U][:] for ds in datasets]
        elif u == DU:
            labelss = [ds.tensors[DU][:, 2] for ds in datasets]
        elif u == D2U:
            labelss = [ds.tensors[D2U][:, 2, 2] for ds in datasets]
        else:
            raise ValueError(f"Invalid label index {u}.")
        for key_id, key_val in zip(
            datasets[0].tensors[IC_KEYS][0],
            datasets[0].tensors[IC_VALUES][0]
            ):
            if key_id != -1:
                key_name = ic_key_str(key_id, "Advection-Reaction-Diffusion")
                titles += f"{key_name} = {key_val:.2f}\n"
        for key_id, key_val in zip(
            datasets[0].tensors[PDE_KEYS][0],
            datasets[0].tensors[PDE_VALUES][0]
            ):
            if key_id != -1:
                key_name = key_str(key_id, "Advection-Reaction-Diffusion")
                titles += f"{key_name} = {key_val:.2f}\n"
        subtitles = [f"t = {ds.tensors[TIMES][0].item():.2f}" for ds in datasets]
    
        del datasets
    
        vmin = min([labels.min() for labels in labelss])
        vmax = max([labels.max() for labels in labelss])
    
        # Plot
        figsize = (len(ts) * 1.6, 1.0)
        plot_points(
            pointss=pointss,
            labelss=labelss,
            nrows=1,
            ncols=len(ts),
            label_name=label_name,
            subtitles=subtitles,
            cmap="inferno",
            figsize=figsize,
            vmin=vmin,
            vmax=vmax,
            save=False,
            dst_file=f"{DATA_DIR}/dataset_{label_name}.png"
        )

In [5]:
u = DU

for i, s in [(2,"t"), (0, "x"), (1, "y")]:
    stddevs_abs = []
    means_abs = []
    stddevs_square = []
    means_square = []
    for sample in [0, 1, 2, 3, 4]:
        datasets = torch.load(FULL_DATASET, weights_only=False).datasets[sample].datasets[1:]
        if u == U:
            labelss_abs = [ds.tensors[U][:].abs() for ds in datasets]
            labelss_square = [ds.tensors[U][:].square() for ds in datasets]
        elif u == DU:
            labelss_abs = [ds.tensors[DU][:, i].abs() for ds in datasets]
            labelss_square = [ds.tensors[DU][:, i].square() for ds in datasets]
        elif u == D2U:
            labelss_abs = [ds.tensors[D2U][:, i, i].abs() for ds in datasets]
            labelss_square = [ds.tensors[D2U][:, i, i].square() for ds in datasets]
        del datasets

        labels_abs = torch.cat(labelss_abs)
        labels_square = torch.cat(labelss_square)

        means_abs.append(labels_abs.mean())
        stddevs_abs.append(labels_abs.std())
        del labels_abs

        means_square.append(labels_square.mean())
        stddevs_square.append(labels_square.std())
        del labels_square

    value = latex_sci(np.mean(means_abs))
    print(f"Mean absolute value ({s}): {value}")
    value = latex_sci(np.std(means_abs))
    print(f"Std dev absolute value ({s}): {value}")

    value = latex_sci(np.mean(means_square))
    print(f"Mean square value ({s}): {value}")
    value = latex_sci(np.std(means_square))
    print(f"Std dev square value ({s}): {value}")

Mean absolute value (t): $1.42 \times 10^{0}$
Std dev absolute value (t): $3.90 \times 10^{-1}$
Mean square value (t): $4.18 \times 10^{0}$
Std dev square value (t): $1.92 \times 10^{0}$
Mean absolute value (x): $3.11 \times 10^{0}$
Std dev absolute value (x): $8.49 \times 10^{-1}$
Mean square value (x): $1.72 \times 10^{1}$
Std dev square value (x): $6.61 \times 10^{0}$
Mean absolute value (y): $2.58 \times 10^{0}$
Std dev absolute value (y): $1.14 \times 10^{0}$
Mean square value (y): $1.45 \times 10^{1}$
Std dev square value (y): $1.11 \times 10^{1}$
